# 7.5. Pooling

Pooling is a simple operation that aggregates information over a _pooling window_, similar to how we perform cross-correlation operations with a kernel over a _convolution window_. The main difference is that with pooling, there are no parameters - it's just a simple, pre-defined operation such as taking the average or maximum value over the pooling window. Pooling helps us _downsample_ the feature map obtained from convolution so minor shifts in detected features such as edges do not affect the end result. For example, it should not matter if we detect an edge on our object of interest at pixel location `(i, j)` vs. `(i + 1, j)` or `(i, j + 1)` - the resulting object is still the same.

Let's briefly go through the 2 most common types of pooling - average pooling vs. max pooling.

In [1]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.8.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 7.5.1. Maximum Pooling and Average Pooling

As their names suggest:

1. Maximum pooling takes the maximum value over the pooling window
1. Average pooling computes the average value over the pooling window

Considering our goal of downsampling and _translation invariance_, max pooling is often the better choice, though average pooling can be seen in early versions of convnets before max pooling was introduced.

An example of a $3 \times 3$ feature map with a max pooling window of $2 \times 2$ illustrated below. We'll use a stride of 1 for our simple example. Denote our feature map by $\mathbf{X}$ and result from pooling by $\mathbf{Y}$.

$$
\begin{align}
\mathbf{X} &= \begin{pmatrix} 0 & 1 & 2 \\ 3 & 4 & 5 \\ 6 & 7 & 8 \end{pmatrix} \\
\mathbf{Y} &= \begin{pmatrix} \max(0, 1, 3, 4) & \max(1, 2, 4, 5) \\ \max(3, 4, 6, 7) & \max(4, 5, 7, 8) \end{pmatrix} \\
&= \begin{pmatrix} 4 & 5 \\ 7 & 8 \end{pmatrix}
\end{align}
$$

Let's implement it from first principles as `pool2d` below. Again, let's ignore the case of multiple channels for now and return to it later.

In [2]:
import mindspore.ops as ops

def pool2d(X, pool_size, mode='max'):
    h, w = X.shape
    p_h, p_w = pool_size
    Y = ops.zeros((h - p_h + 1, w - p_w + 1))
    pool_op = None
    match mode:
        case 'max': # Max. pooling
            pool_op = lambda X: ops.max(X)[0]
        case 'avg': # Avg. pooling
            pool_op = ops.mean
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = pool_op(X[i:i+p_h, j:j+p_w])
    return Y

Let's use our example above to verify our `pool2d` function is implemented correctly.

In [3]:
X = ops.reshape(ops.arange(9), (3, 3))
X

/usr/local/Ascend/cann-8.5.0/python/site-packages/asc_op_compile_base/asc_op_compiler/ascendc_compile_gen_code.py:161: SyntaxWarning: invalid escape sequence '\w'
  match = re.search(f'{option}=(\w+)', ' '.join(compile_options))
2026-05-03 10:05:56.525976: E external/org_tensorflow/tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute is_closed which is not in the op definition: Op<name=Range; signature=start:Tidx, limit:Tidx, delta:Tidx -> output:Tidx; attr=Tidx:type,default=DT_INT32,allowed=[DT_BFLOAT16, DT_HALF, DT_FLOAT, DT_DOUBLE, DT_INT8, DT_INT16, DT_INT32, DT_INT64, DT_UINT16, DT_UINT32]> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node Range1}}


Tensor(shape=[3, 3], dtype=Int64, value=
[[0, 1, 2],
 [3, 4, 5],
 [6, 7, 8]])

In [4]:
from mindspore import dtype as mstype

Y = pool2d(X, (2, 2)).astype(dtype=mstype.int64)
Y

Tensor(shape=[2, 2], dtype=Int64, value=
[[4, 5],
 [7, 8]])

Let's try average pooling as well.

In [5]:
Y = pool2d(X, (2, 2), mode='avg').astype(dtype=mstype.int64)
Y

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result


Tensor(shape=[2, 2], dtype=Int64, value=
[[2, 3],
 [5, 6]])

## 7.5.2. Padding and Stride

As with convolutional layers, we can specify padding and stride for our pooling layers. In MindSpore, the maximum pooling layer is provided as [`mindspore.nn.MaxPool2d`](https://www.mindspore.cn/docs/en/r2.8.0/api_python/nn/mindspore.nn.MaxPool2d.html).

In [6]:
X = ops.reshape(ops.arange(16).astype(dtype=mstype.float16), (1, 1, 4, 4))
X

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result


Tensor(shape=[1, 1, 4, 4], dtype=Float16, value=
[[[[ 0.0000e+00,  1.0000e+00,  2.0000e+00,  3.0000e+00],
   [ 4.0000e+00,  5.0000e+00,  6.0000e+00,  7.0000e+00],
   [ 8.0000e+00,  9.0000e+00,  1.0000e+01,  1.1000e+01],
   [ 1.2000e+01,  1.3000e+01,  1.4000e+01,  1.5000e+01]]]])

We set the shape of our tensor as `(1, 1, 4, 4)` above to accomodate for the batch size and channel dimension respectively.

Let's apply max pooling to our input tensor `X` with a pooling window of $3 \times 3$. By default, the stride is 1.

In [7]:
import mindspore.nn as nn

max_pool2d = nn.MaxPool2d(kernel_size=3)
Y = max_pool2d(X)
Y, Y.shape

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: Sy

(Tensor(shape=[1, 1, 2, 2], dtype=Float16, value=
 [[[[ 1.0000e+01,  1.1000e+01],
    [ 1.4000e+01,  1.5000e+01]]]]),
 (1, 1, 2, 2))

Let's apply a stride size of 3 instead.

In [8]:
max_pool2d = nn.MaxPool2d(kernel_size=3, stride=3)
Y = max_pool2d(X)
Y, Y.shape

(Tensor(shape=[1, 1, 1, 1], dtype=Float16, value=
 [[[[ 1.0000e+01]]]]),
 (1, 1, 1, 1))

As with convolutional layers, we can set the `pad_mode` and `padding` as well.

In [9]:
max_pool2d = nn.MaxPool2d(kernel_size=3, stride=3, pad_mode='same')
Y = max_pool2d(X)
Y, Y.shape

(Tensor(shape=[1, 1, 2, 2], dtype=Float16, value=
 [[[[ 5.0000e+00,  7.0000e+00],
    [ 1.3000e+01,  1.5000e+01]]]]),
 (1, 1, 2, 2))

As with convolutional layers, we can also specify rectangular pooling window, padding and stride sizes - they don't always have to be perfect squares!

In [10]:
max_pool2d = nn.MaxPool2d((2, 3), stride=(2, 3), pad_mode='pad', padding=(0, 1))
Y = max_pool2d(X)
Y, Y.shape

(Tensor(shape=[1, 1, 2, 2], dtype=Float16, value=
 [[[[ 5.0000e+00,  7.0000e+00],
    [ 1.3000e+01,  1.5000e+01]]]]),
 (1, 1, 2, 2))

## 7.5.3. Multiple Channels

It wouldn't be incredibly useful if our pooling layer could only deal with images containing exactly 1 input channel. Fortunately, as with convolutional layers, MindSpore's built-in pooling layers support multiple input channels as well.

Unlike convolutional layers where the number of input and output channels can both be specified, pooling layers always return the same number of channels as its input. Furthermore, instead of summing the results across multiple input channels, for pooling we consider each channel separately and apply the pooling operation per channel.

Let's concatenate `X` and `X + 1` along the 1st dimension to create our feature map of 2 channels and observe how our max pooling layer simply applies the pooling operation to each individual channel in isolation, returning a transformed feature map with 2 channels.

In [11]:
X = ops.cat([X, X + 1])
X, X.shape

(Tensor(shape=[2, 1, 4, 4], dtype=Float16, value=
 [[[[ 0.0000e+00,  1.0000e+00,  2.0000e+00,  3.0000e+00],
    [ 4.0000e+00,  5.0000e+00,  6.0000e+00,  7.0000e+00],
    [ 8.0000e+00,  9.0000e+00,  1.0000e+01,  1.1000e+01],
    [ 1.2000e+01,  1.3000e+01,  1.4000e+01,  1.5000e+01]]],
  [[[ 1.0000e+00,  2.0000e+00,  3.0000e+00,  4.0000e+00],
    [ 5.0000e+00,  6.0000e+00,  7.0000e+00,  8.0000e+00],
    [ 9.0000e+00,  1.0000e+01,  1.1000e+01,  1.2000e+01],
    [ 1.3000e+01,  1.4000e+01,  1.5000e+01,  1.6000e+01]]]]),
 (2, 1, 4, 4))

In [12]:
max_pool2d = nn.MaxPool2d(3, pad_mode='pad', padding=1, stride=2)
Y = max_pool2d(X)
Y, Y.shape

(Tensor(shape=[2, 1, 2, 2], dtype=Float16, value=
 [[[[ 5.0000e+00,  7.0000e+00],
    [ 1.3000e+01,  1.5000e+01]]],
  [[[ 6.0000e+00,  8.0000e+00],
    [ 1.4000e+01,  1.6000e+01]]]]),
 (2, 1, 2, 2))

## 7.5.4. Summary

We saw in this chapter how pooling helps us aggregate features to increase the receptive field of our feature maps relative to the original input image. This process is known as _downsampling_.

With the basic building blocks of convolutional neural networks \(CNNs\) covered, let's take a look at a real, complete example of a convnet in our next chapter - LeNet.